# The detector

This notebook runs the equations of [Reference frames](../theory/frames.rst), sections *Angles of a diffracted ray* and *Detector frame*: how a pixel $(s, f)$ sits in the lab, how a diffracted ray finds its pixel, and the $(2\theta, \eta)$ of every pixel.
It ends with FABLE's test case, from a crystal to a pixel.

![The detector seen from the sample](../theory/figures/frames_detector.svg)

In [ ]:
import anri.utils

anri.utils.setup()  # before any JAX computation: don't grab most of the GPU memory at start-up

import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt

import anri.io
from anri.diffract.scatter import k_out_to_tth_eta, tth_eta_to_k_out
from anri.geom import detector

## From ImageD11 parameters

A detector is described by ImageD11's parameters: the beam centre in raw (slow, fast) pixels, the pixel sizes, the distance from the rotation axis, three tilts (radians) and the orientation matrix $o$.
Here, a 1000 x 1000 detector of 150 µm pixels, 200 mm from the sample, with the beam near its middle (lengths in µm: any unit will do, as long as the sample positions use it too):

In [ ]:
pars = {
    "z_center": 480.0, "y_center": 520.0,  # beam centre, (slow, fast) pixels
    "z_size": 150.0, "y_size": 150.0,  # pixel size
    "distance": 200e3,
    "tilt_x": 0.0, "tilt_y": 0.0, "tilt_z": 0.0,
    "o11": 1, "o12": 0, "o21": 0, "o22": 1,
}  # fmt: skip
shape = (1000, 1000)
det = anri.io.imaged11.detector_from_pars(pars)
{k: det[k] for k in ("s_step_lab", "f_step_lab", "det_origin_lab")}

`detector_from_pars` reduces them to three vectors in the lab: one pixel along slow, one along fast, and the position of pixel (0, 0).
With the identity $o$ and no tilts, slow runs up (lab $+z$) and fast to the left looking downstream (lab $+y$), so pixel (0, 0) is at the lower right, and the beam centre is $L$ along the beam:

In [ ]:
def pixel_to_lab(s, f, det):
    """Lab position of pixel (s, f): pixel (0, 0) plus s slow steps plus f fast steps."""
    return det["det_origin_lab"] + s * det["s_step_lab"] + f * det["f_step_lab"]


pixel_to_lab(pars["z_center"], pars["y_center"], det)

`det_to_lab` gives the same from the full transform of equation [(frm-det)](../theory/frames.rst), and `lab_to_det` inverts it:

In [ ]:
s, f = 123.4, 789.0
v_lab = detector.det_to_lab(s, f, det["det_trans"], det["beam_cen_shift"], det["x_distance_shift"])
(
    v_lab,
    pixel_to_lab(s, f, det),
    detector.lab_to_det(*v_lab, det["det_trans"], det["beam_cen_shift"], det["x_distance_shift"]),
)

A diffracted ray leaves the sample along $\vec{k}_{\text{out}}$; `raytrace_to_det` finds the pixel where it meets the detector plane.
The ray from the lab origin through that pixel finds it again:

In [ ]:
detector.raytrace_to_det(v_lab, jnp.zeros(3), det["s_step_lab"], det["f_step_lab"], det["det_origin_lab"])

## $2\theta$ and $\eta$ of every pixel

Seen from the lab origin, each pixel has a scattering angle $2\theta$ and an azimuth $\eta$ about the beam (equation [(frm-eta)](../theory/frames.rst)): $\eta = 0$ straight up, $90°$ at 3 o'clock looking downstream (towards lab $-y$), $\pm 180°$ straight down.
The functions act on arrays of directions, so a whole detector is one call:

In [ ]:
k_in = jnp.array([1.0, 0.0, 0.0])


def pixel_angles(det, shape):
    s, f = jnp.meshgrid(jnp.arange(shape[0]), jnp.arange(shape[1]), indexing="ij")
    return k_out_to_tth_eta(pixel_to_lab(s[..., None], f[..., None], det), k_in)


def show(ax, tth, eta, title):
    """The raw image as stored, pixel (0, 0) at the bottom left: eta in colour, rings of constant 2theta."""
    im = ax.imshow(eta, origin="lower", cmap="twilight_shifted", vmin=-180, vmax=180)
    ax.contour(tth, levels=np.arange(1.0, 30.0, 2.0), colors="k", linewidths=0.6)
    ax.set(title=title, xlabel="fast", ylabel="slow")
    return im


tth, eta = pixel_angles(det, shape)
fig, ax = plt.subplots(figsize=(6, 5))
im = show(ax, tth, eta, "identity o-matrix")
fig.colorbar(im, ax=ax, label=r"$\eta$ (degrees)")
plt.show()

Stored with pixel (0, 0) at the bottom left, this image is the detector seen from *behind* (from downstream, looking back at the sample): lab $+y$ (fast) is to the right.
So $\eta = 0$ (pale) is up, and $\eta = 90°$, towards lab $-y$, is on the left.

## The orientation matrix

Detectors store their images in different ways. The $o$-matrix (equation [(frm-omatrix)](../theory/frames.rst)) says how the (slow, fast) array sits on the detector; its eight choices flip or transpose the image.
The same detector, read out with fast running the other way ($o_{22} = -1$) and transposed ($o_{12} = o_{21} = 1$, $o_{11} = o_{22} = 0$):

In [ ]:
flipped = anri.io.imaged11.detector_from_pars({**pars, "o22": -1})
transposed = anri.io.imaged11.detector_from_pars(
    {**pars, "o11": 0, "o12": 1, "o21": 1, "o22": 0, "z_center": 520.0, "y_center": 480.0}
)
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
show(axs[0], *pixel_angles(flipped, shape), r"$o_{22} = -1$")
show(axs[1], *pixel_angles(transposed, shape), "transposed")
plt.show()
flipped["f_step_lab"], transposed["s_step_lab"], transposed["f_step_lab"]

With $o_{22} = -1$ the fast step points to lab $-y$, so the image is mirrored left to right; transposed, slow and fast swap their lab directions.
The beam centre is given in the raw pixels of each, so for the transposed one (slow, fast) = (520, 480).

## Tilts

The tilts turn the detector about the lab axes ($\matr{R}_x \matr{R}_y \matr{R}_z$, right-handed), about the point where the beam meets it.
A positive `tilt_y` swings the top of the detector downstream, away from the sample, so the rings crowd at the bottom, where the detector is nearer.
A positive `tilt_z` swings its lab $+y$ side (the right of the raw image) towards the sample, so the rings crowd on the right.
`tilt_x` turns the detector about the beam, and with it the $\eta$ of every pixel:

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, tilt in zip(axs, ("tilt_x", "tilt_y", "tilt_z")):
    tilted = anri.io.imaged11.detector_from_pars({**pars, tilt: np.radians(25.0)})
    show(ax, *pixel_angles(tilted, shape), f"{tilt} = 25 degrees")
plt.show()

## FABLE's test case

FABLE's geometry document ends with a test case (its pages 10-11), which [Reference frames](../theory/frames.rst) quotes: an fcc grain with $a = 4.05$ Å and a given $\matr{U}$, at sample position $(-0.0602, 0.215, 0)$ mm, a 69.533 keV beam, and a tilted detector of 4.3 µm pixels.
The parameters below are FABLE's "detector tilts I", ImageD11's, in mm.
FABLE converts energy to wavelength with $hc = 12.398$ keV Å, where Anri uses CODATA's 12.39842; with FABLE's, the angles agree to its last digits.

In [ ]:
hc, energy, a = 12.398, 69.533, 4.05
wavelength = hc / energy
U = jnp.array([[0.872986, 0.436832, -0.216965], [-0.334822, 0.860185, 0.384678], [0.354669, -0.263174, 0.897190]])
B = jnp.eye(3) / a  # a cubic cell
position = jnp.array([-0.0602, 0.215, 0.0])
hkl = jnp.array([-2.0, -2.0, 2.0])
fable_pars = {
    "distance": 9.284758, "y_center": 724.953252, "z_center": 531.210607, "y_size": 0.0043, "z_size": 0.0043,
    "tilt_x": np.radians(1.0988), "tilt_y": np.radians(2.085), "tilt_z": np.radians(3.473),
    "o11": 1, "o12": 0, "o21": 0, "o22": 1,
}  # fmt: skip
fdet = anri.io.imaged11.detector_from_pars(fable_pars)

`anri.fwd.peak.get_centroid` runs the whole chain: $\vec{g} = \matr{U}\matr{B}\vec{h}$, the $\omega$ that satisfies the Ewald condition, the diffracted ray from the grain's position at that $\omega$, and its pixel.
Its arguments are the UBI, the position, $\vec{h}$, which of the two $\omega$ solutions, the beam, the goniometer and the detector:

In [ ]:
from anri.fwd.peak import get_centroid, hkl_to_k_omega

ubi = jnp.linalg.inv(U @ B)
(sc, fc, omega), valid = get_centroid(
    ubi, position, hkl, 1.0, wavelength, k_in, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    fdet["s_step_lab"], fdet["f_step_lab"], fdet["det_origin_lab"], False,
)  # fmt: skip
_, k_out, _, _ = hkl_to_k_omega(ubi, hkl, 1.0, wavelength, k_in, 0.0, 0.0, 0.0, 0.0)
tth, eta = k_out_to_tth_eta(k_out, k_in)
print(f"omega {float(omega):.6f}  (FABLE 79.793676)")
print(f"theta {float(tth) / 2:.6f}   (FABLE 4.373314)")
print(f"eta   {float(eta):.6f}  (FABLE 62.190963)")
print(f"pixel (y_det, z_det) = ({float(fc):.1f}, {float(sc):.1f})  (FABLE (418, 698), whole pixels)")

The $10^{-5}$ degrees left in $\eta$ is FABLE's $\matr{U}$, printed to six digits.
`tests/unit/geom/test_fable_case.py` checks the same numbers in CI.

The [simple forward model](forward_model_simple.ipynb) does this for every reflection of a grain.